In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        if requirement is None:  # a GPU-only package on a CPU runtime
            continue
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, cpu_requirement, _, match) in pinned.items():
    if not gpu and cpu_requirement is None:
        continue
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "85bd815efd86af24338a0f0c45844cb7e1e6f155"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/85bd815efd86af24338a0f0c45844cb7e1e6f155/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# ConvNeXt: Modernizing a ResNet Step by Step

Every architecture in this chapter so far has been convolutional, and until
2020 convnets sat atop every major image-classification leaderboard. In 2021,
however, an architecture without a single standard spatial convolution took
the lead: the *Swin Transformer* [@liu2021swin], built around
*attention*, achieved 81.3% top-1 accuracy on ImageNet compared with 76.1% for
a similarly sized ResNet-50. Did attention itself cause that five-point leap—and
if so, were convolutional networks obsolete?

A brief sketch of vision transformers suffices here (we develop them fully in
that section). A *vision transformer*
[@Dosovitskiy.Beyer.Kolesnikov.ea.2021] partitions an image into a grid
of patches, embeds each patch as a feature vector, and processes the sequence
through a stack of identical blocks. Each block has two sublayers: an
*attention* layer that lets every patch compute a data-dependent weighted
average of other patches, followed by a pointwise MLP applied independently
to each patch vector. Swin restores hierarchical convnet structure by
computing attention inside local $7 \times 7$ windows and downsampling
between stages.

After that section, we should immediately be skeptical of
crediting a five-point gap to attention alone. Swin differs from a 2015 ResNet
in at least a dozen ways beyond attention: it trains with a 300-epoch modern
recipe, slices images with a patch stem, allocates depth differently across
stages, and adopts different activations and normalizations. To disentangle
these factors, @liu2022convnet ran a landmark controlled experiment:
starting from a standard ResNet-50, they introduced Swin-inspired
modernizations one at a time, measuring ImageNet top-1 accuracy after every
change without ever adding attention. The resulting pure convnet, *ConvNeXt*,
reaches 82.0%—edging past Swin-T using primitives we have already built in
this chapter: modern training recipes (that section),
depthwise convolutions and inverted bottlenecks (that section),
and layer normalization (that section). Because each row of their
ablation requires a full ImageNet run, we walk through their published roadmap
step by step, implement ConvNeXt from scratch, and then conduct a compact
three-step modernization roadmap of our own on Fashion-MNIST starting from a
parameter-matched ResNet-18.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

## The Modernization Roadmap

the table walks through the exact sequence of changes
tested by @liu2022convnet. Each row accumulates every modification
above it while keeping compute anchored near Swin-T's budget of roughly 4.5
billion multiply-adds per image (reported as 4.5 GFLOPs in the paper).

<!-- tbl-caption:The ConvNeXt modernization roadmap [@liu2022convnet]. ImageNet top-1 accuracy of a ResNet-50 as design changes accumulate; each row includes all rows above it, at roughly constant cost. Swin-T, a transformer of similar cost, reaches 81.3%. -->

| Change | What it means | Top-1 |
|:--|:--|:--|
| (starting point) | ResNet-50 with its 2015 recipe | 76.1% |
| Modern training recipe | 300 epochs, AdamW, Mixup, label smoothing, stochastic depth (that section) | 78.8% |
| Stage ratio | blocks per stage $(3, 4, 6, 3) \to (3, 3, 9, 3)$ | 79.4% |
| Patchify stem | $7 \times 7$ stride-2 convolution plus pooling $\to$ one $4 \times 4$ stride-4 convolution | 79.5% |
| Depthwise convolutions | the $3 \times 3$ convolution becomes depthwise; width grows from 64 to 96 channels | 80.5% |
| Inverted bottleneck | expand $4\times$ inside the block instead of compressing | 80.6% |
| Large kernels | depthwise convolution moved to the front and enlarged to $7 \times 7$ | 80.6% |
| ReLU $\to$ GELU | a smooth activation, as in transformers | 80.6% |
| Fewer activations | one activation per block instead of three | 81.3% |
| Fewer normalizations | one normalization per block instead of three | 81.4% |
| BN $\to$ LN | layer normalization over channels at each position | 81.5% |
| Separate downsampling | an LN plus $2 \times 2$ stride-2 convolution between stages | 82.0% |

Immediately, the very first row delivers the largest single jump—without
touching a single layer of the network. Echoing that section,
upgrading the training recipe alone lifts an untouched ResNet-50 by 2.7
percentage points. Of the 5.9-point total gain separating 2015 ResNet-50 from
ConvNeXt, 2.7 points come from training and 3.2 from architecture (in this
ordering), confirming that pitting a modern transformer against a 2015-recipe
ResNet confounded recipes with architectures.

### Macro Design

The next two rows reshape the network's macro structure. Swin-T allocates its
blocks across four stages in a $1{:}1{:}3{:}1$ ratio, concentrating depth in
stage 3 where feature maps are compact and compute is cheap; shifting
ResNet-50's $(3, 4, 6, 3)$ layout to $(3, 3, 9, 3)$ adds 0.6 points. Next
comes the stem. Whereas ResNet opens with an overlapping $7 \times 7$ stride-2
convolution followed by max-pooling to downsample fourfold, a vision
transformer chops the image into disjoint patches and projects each patch
through a shared linear layer—which is algebraically identical to a
non-overlapping $4 \times 4$ convolution with stride 4 (that section).
Swapping in this *patchify* stem preserves accuracy (79.5%) while producing
one token vector per $4 \times 4$ patch, just like a transformer.

### Depthwise Convolutions and the Inverted Bottleneck

A transformer block cleanly factors computation into two stages: attention
mixes information *across spatial positions*, and the MLP mixes information
*across channels* at each position. As we saw in
that section, a convnet can perform the exact same
factorization: a *depthwise* convolution mixes across space within each
channel, and $1 \times 1$ convolutions mix across channels at each pixel.
Replacing the dense $3 \times 3$ layer in ResNet's bottleneck with a depthwise
convolution slashes compute enough to widen stage 1 from 64 to 96 channels
(matching Swin-T), jumping accuracy to 80.5%.

Once the spatial convolution is depthwise rather than dense, its cost scales
linearly rather than quadratically with width, removing the rationale for
compressing channels before filtering. ConvNeXt therefore adopts MobileNetV2's
*inverted bottleneck* (that section), maintaining a thin
residual stream and expanding $4\times$ inside the block—the exact same
$1\times \to 4\times \to 1\times$ geometry as a transformer MLP—nudging
accuracy to 80.6%.

With spatial filtering now lightweight, kernel size also becomes easy to
enlarge. First, moving the depthwise convolution to the front of the block so
it acts on $c$ channels instead of $4c$ drops accuracy by 0.7 points (to
79.9%—the sole dip on the roadmap); expanding its kernel from $3 \times 3$ to
$7 \times 7$ immediately recovers that ground. A $7 \times 7$ filter matches
the receptive field of three stacked $3 \times 3$ layers
(that section). Back in VGG (that section), dense $7 \times 7$
kernels were prohibitively expensive ($k^2 c_\textrm{i} c_\textrm{o}$); for a
depthwise kernel ($k^2 c$), growing from $3 \times 3$ to $7 \times 7$ only
scales a tiny fraction of block compute by $49/9 \approx 5$. Accuracy
saturates right at $7 \times 7$—matching the exact window size of Swin's local
attention.

### Micro Design

The final group of edits refines individual layers inside the block. Swapping
ReLU for *GELU* [@Hendrycks.Gimpel.2016]—the smooth activation favored by
transformers—leaves accuracy unchanged on its own (80.6%). What *does* move
the needle is trimming the *number* of activations and normalizations. A
classic ResNet bottleneck puts a normalization and an activation after all
three convolutions, whereas a transformer block uses only a single activation
inside its MLP and fewer normalizations. Removing two of the three GELUs—keeping
only the one between the two $1 \times 1$ layers—jumps accuracy by 0.7 points,
the biggest micro-design gain on the table. (There is still no crisp
theoretical proof for why removing two nonlinearities improves accuracy; it is
best viewed as a clean empirical finding.) Trimming three normalizations down
to one adds 0.1 points, and switching that lone normalization from batch
normalization to per-position *layer normalization* across channels
(that section) adds another 0.1 points (81.5%).
the figure compares the original ResNet bottleneck
against the resulting ConvNeXt block (including the learnable per-channel
layer scale $\boldsymbol{\gamma}$ described below).

![The ResNet-50 bottleneck block and the ConvNeXt block that the roadmap turns it into. Three normalizations and three activations become one of each; the compress-process-expand bottleneck becomes an inverted bottleneck led by a $7 \times 7$ depthwise convolution, and a learnable per-channel layer scale $\boldsymbol{\gamma}$ multiplies the branch before the addition.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/arch-resnet-vs-convnext-block.svg)

Finally, whereas ResNet downsamples inside the opening block of each stage
with a stride-2 convolution, Swin inserts dedicated patch-merging layers
between stages. Adopting separate downsampling layers—a layer normalization
followed by a $2 \times 2$ stride-2 convolution between stages, plus a
normalization after the stem and before the linear head to stabilize
training—completes the roadmap at 82.0%. the figure shows the
assembled ConvNeXt-T backbone, stacking $(3, 3, 9, 3)$ blocks of widths
$(96, 192, 384, 768)$ in the classic stem--body--head layout
(that section).

![ConvNeXt-T. A patchify stem replaces convolution-plus-pooling; stages of 3, 3, 9, and 3 blocks are joined by separate LN plus $2 \times 2$ stride-2 downsampling layers; there is no pooling anywhere in the body.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/arch-convnext.svg)

At matched compute, ConvNeXt-T surpasses Swin-T without a single attention
layer. The takeaway is not that convolutions are inherently superior to
attention, but that the 2021 accuracy gap came from training recipes and
architectural design hygiene rather than attention alone—under matched tuning,
both paradigms scale comparably.

## Implementation

Because the modernization roadmap stripped away more layers than it added, the
ConvNeXt block is remarkably clean to implement.

### The ConvNeXt Block

Each block applies a $7 \times 7$ depthwise convolution, per-position layer
normalization, a $1 \times 1$ expansion ($c \to 4c$), GELU, and a $1 \times 1$
projection ($4c \to c$), then adds the residual input. Two training
stabilizers from the paper accompany the block. *Layer scale*
[@touvron2021cait] multiplies the residual branch by a learnable
per-channel vector $\boldsymbol{\gamma}$ initialized to $10^{-6}$ (distinct
from batch-norm scale), ensuring every block starts nearly as the identity so
deep optimization begins smoothly. *Stochastic depth* randomly drops entire
residual branches during training via `d2l.drop_path`
(that section).

Because layer normalization operates across the channel vector at each spatial
location, in a channels-last tensor `(batch, height, width, channels)` it is
standard `LayerNorm` over the final axis, and the two $1 \times 1$
convolutions are ordinary linear layers over that same axis. Following
@liu2022convnet, we initialize all convolutional and linear weights
from $\mathcal{N}(0, 0.02^2)$ with zero biases.
<!-- d2l:prose id=convnext-md-3 fw=tensorflow, jax -->

Images are stored channels-last, so the block uses the library's layer
normalization and dense layers directly.

In [ ]:
def layer_norm(dim, rngs):
    return nnx.LayerNorm(dim, epsilon=1e-6, rngs=rngs)

class ConvNeXtBlock(nnx.Module):
    """Depthwise 7x7, norm, 1x1 expand, GELU, 1x1 project, scaled residual."""
    def __init__(self, dim, drop_prob=0.0, layer_scale=1e-6, norm=layer_norm,
                 rngs=None):
        rngs = (nnx.Rngs(params=d2l.get_key(), dropout=d2l.get_key())
                if rngs is None else rngs)
        w_init = nnx.initializers.normal(0.02)
        self.dwconv = nnx.Conv(dim, dim, kernel_size=(7, 7), padding='same',
                               feature_group_count=dim, kernel_init=w_init,
                               rngs=rngs)
        self.norm = norm(dim, rngs)
        self.pwconv1 = nnx.Linear(dim, 4 * dim, kernel_init=w_init,
                                  rngs=rngs)
        self.pwconv2 = nnx.Linear(4 * dim, dim, kernel_init=w_init,
                                  rngs=rngs)
        self.gamma = nnx.Param(layer_scale * jnp.ones(dim))
        self.drop_prob, self.rngs = drop_prob, rngs
        self.deterministic = False

    def set_view(self, *, deterministic):
        self.deterministic = deterministic

    def __call__(self, X):
        Y = self.norm(self.dwconv(X))
        Y = self.pwconv2(nnx.gelu(self.pwconv1(Y), approximate=False))
        return X + d2l.drop_path(self.rngs.dropout(), self.gamma * Y,
                                 self.drop_prob, not self.deterministic)

### The Full Network

As in VGG (that section), the complete architecture iterates over a tuple
of `(depth, channels)` stage specifications, opening with the patchify stem
plus layer normalization and inserting dedicated downsampling layers between
stages with no spatial pooling anywhere in the body. Stochastic-depth drop
probabilities ramp linearly from 0 in the first block to `drop_path_max` in
the last. For Fashion-MNIST we adopt the compact *ConvNeXt-atto* profile—widths
$(40, 80, 160, 320)$ and stage depths $(2, 2, 6, 2)$—keeping the third stage
deepest just like ConvNeXt-T. Exposing `norm` as an argument and `stem` as a
method makes swapping them trivial in our ablation below.

In [ ]:
class ConvNeXt(d2l.Classifier):
    def __init__(self, arch=((2, 40), (2, 80), (6, 160), (2, 320)),
                 lr=2e-3, num_classes=10, drop_path_max=0.1, norm=layer_norm,
                 rngs=None):
        super().__init__()
        self.save_hyperparameters(ignore=['rngs'])
        rngs = (nnx.Rngs(params=d2l.get_key(), dropout=d2l.get_key(),
                         mixup=d2l.get_key()) if rngs is None else rngs)
        self.rngs = rngs
        w_init = nnx.initializers.normal(0.02)
        depths = [d for d, c in arch]
        rates = jnp.linspace(0, drop_path_max, sum(depths)).tolist()
        layers = self.stem(arch[0][1], norm, rngs)
        c_prev, b = arch[0][1], 0
        for i, (depth, c) in enumerate(arch):
            if i > 0:  # Separate downsampling layer between stages
                layers += [norm(c_prev, rngs),
                           nnx.Conv(c_prev, c, kernel_size=(2, 2),
                                    strides=(2, 2), kernel_init=w_init,
                                    rngs=rngs)]
            for _ in range(depth):
                layers.append(ConvNeXtBlock(c, rates[b], norm=norm, rngs=rngs))
                b += 1
            c_prev = c
        layers += [global_avg_pool, norm(c_prev, rngs),
                   nnx.Linear(c_prev, num_classes, kernel_init=w_init,
                              rngs=rngs)]
        self.net = nnx.Sequential(*layers)

    def stem(self, c, norm, rngs):  # Patchify: a 4x4 convolution, stride 4
        return [nnx.Conv(1, c, kernel_size=(4, 4), strides=(4, 4),
                         kernel_init=nnx.initializers.normal(0.02),
                         rngs=rngs),
                norm(c, rngs)]

def global_avg_pool(X):  # Average over height and width (NHWC)
    return X.mean(axis=(1, 2))

On $64 \times 64$ Fashion-MNIST inputs, the patchify stem produces a
$16 \times 16$ grid, and the three downsampling layers step resolution down to
$8 \times 8$, $4 \times 4$, and $2 \times 2$ before global average pooling.
Let us check the output shape and count trainable parameters.

In [ ]:
model = ConvNeXt()
assert model(jnp.zeros((1, 64, 64, 1))).shape == (1, 10)
d2l.num_params(model)

Our ConvNeXt-atto model contains exactly 3,376,450 parameters—roughly a third
of ResNet-18's 11.2 million (that section). Because any misconfigured
width or kernel size immediately shifts this exact number, matching the
parameter count is a reliable sanity check for any reimplementation.

## A Small Roadmap on Fashion-MNIST

Every choice in the table was validated on 1.3 million
$224 \times 224$ ImageNet photos. Do those same architectural decisions carry
over to small $64 \times 64$ grayscale garments? To find out, we run a compact
cumulative roadmap of our own: starting from a parameter-matched control—a
ResNet-18 narrowed from base width 64 to 35 channels (3.35M parameters)—we
apply ConvNeXt's modifications in three cumulative steps. Every model trains
for 20 epochs on full Fashion-MNIST under the compact modern recipe from
that section (AdamW with decoupled weight decay, 10% linear
warmup plus cosine decay, label smoothing, and Mixup; that section
covers AdamW in detail). To isolate architecture cleanly, we set ConvNeXt's
stochastic-depth rate to 0 so both families share identical regularization.

Packaging the training procedure as a `ModernRecipe` mixin lets every model in
the roadmap inherit the exact same optimizer, schedule, and mixed training
step while keeping validation loss as plain cross-entropy.

In [ ]:
class ModernRecipe:
    """AdamW, linear warmup then cosine decay, label smoothing, Mixup."""
    def configure_optimizers(self):
        warmup = self.num_steps // 10
        schedule = lambda t: self.lr * d2l.cosine_warmup(t, self.num_steps,
                                                         warmup)
        return optax.adamw(schedule, weight_decay=0.05)

    def training_step(self, batch):
        X, y_a, y_b, lam = d2l.mixup(self.rngs.mixup(), *batch, alpha=0.2)
        Y_hat = self(X)
        smooth = lambda y: optax.smooth_labels(
            jax.nn.one_hot(y, Y_hat.shape[1]), 0.1)
        loss = lambda y: optax.softmax_cross_entropy(Y_hat, smooth(y)).mean()
        return lam * loss(y_a) + (1 - lam) * loss(y_b)

class ModernConvNeXt(ModernRecipe, ConvNeXt):
    def __init__(self, num_steps, lr=2e-3, norm=layer_norm):
        super().__init__(lr=lr, drop_path_max=0.0, norm=norm)
        self.num_steps = num_steps

Our baseline `CompactResNet18` stacks `d2l.Residual` blocks
(that section) across widths 35, 70, 140, and 280 behind ResNet's
standard $7 \times 7$ stride-2 stem and max-pooling.

In [ ]:
class CompactResNet18(ModernRecipe, d2l.Classifier):
    """ResNet-18 with base width 35 instead of 64, trained by the recipe."""
    def __init__(self, num_steps, lr=2e-3, num_classes=10, base=35,
                 rngs=None):
        super().__init__()
        self.save_hyperparameters(ignore=['rngs'])
        rngs = (nnx.Rngs(params=d2l.get_key(), mixup=d2l.get_key())
                if rngs is None else rngs)
        self.rngs = rngs
        w_init = nnx.initializers.variance_scaling(2.0, 'fan_in', 'normal')
        layers = [nnx.Conv(1, base, (7, 7), strides=(2, 2), padding=3,
                           use_bias=False, kernel_init=w_init, rngs=rngs),
                  nnx.BatchNorm(base, momentum=0.9, use_fast_variance=False,
                                rngs=rngs), nnx.relu, max_pool]
        c_prev = base
        for i, c in enumerate((base, 2 * base, 4 * base, 8 * base)):
            for j in range(2):
                down = i > 0 and j == 0
                layers.append(d2l.Residual(
                    c, use_1x1conv=down, strides=(2, 2) if down else (1, 1),
                    in_channels=c_prev, rngs=rngs))
                c_prev = c
        layers += [global_avg_pool,
                   nnx.Linear(c_prev, num_classes, kernel_init=w_init,
                              rngs=rngs)]
        self.net = nnx.Sequential(*layers)

def max_pool(X):  # 3x3 windows, stride 2, padding 1
    return nnx.max_pool(X, (3, 3), (2, 2), padding=((1, 1), (1, 1)))

We bundle the paper's changes into three clean steps by architectural region:
(1) replace ResNet's blocks and stages with ConvNeXt's $7 \times 7$ inverted
bottlenecks, $(2, 2, 6, 2)$ stage ratio, single GELU/norm, layer scale, and
separate downsampling while keeping ResNet's stem and batch normalization
(covering eight rows of the table); (2) swap batch
normalization for layer normalization; and (3) replace ResNet's stem with the
$4 \times 4$ patchify stem to arrive at full ConvNeXt. (Although the original
paper swapped the stem earlier, the final architecture is order-independent,
even if individual step attributions can vary slightly.) Overriding `stem` in
`ResNetStemConvNeXt` and toggling `norm` implements all four models cleanly.

In [ ]:
def batch_norm(dim, rngs):  # Decay 0.9 and a two-pass variance, as in ResNet
    return nnx.BatchNorm(dim, momentum=0.9, use_fast_variance=False,
                         rngs=rngs)

class ResNetStemConvNeXt(ModernConvNeXt):
    """ConvNeXt with the stem of ResNet-18 instead of the patchify stem."""
    def stem(self, c, norm, rngs):  # 7x7 convolution, stride 2, max-pooling
        return [nnx.Conv(1, c, kernel_size=(7, 7), strides=(2, 2), padding=3,
                         kernel_init=nnx.initializers.normal(0.02),
                         rngs=rngs),
                norm(c, rngs), nnx.relu, max_pool]

roadmap = {'compact ResNet-18': CompactResNet18,
           'ConvNeXt blocks and stages, BN': (
               lambda num_steps: ResNetStemConvNeXt(num_steps,
                                                    norm=batch_norm)),
           '+ BN -> LN': ResNetStemConvNeXt,
           '+ patchify stem = ConvNeXt': ModernConvNeXt}

Let us train all four models in sequence and record their parameter counts and
final validation accuracies.

In [ ]:
data = d2l.FashionMNIST(batch_size=128, resize=(64, 64))
num_steps = 20 * len(data.train_dataloader())
results = {}
for name, make_model in roadmap.items():
    model = make_model(num_steps)
    trainer = d2l.Trainer(max_epochs=20, num_gpus=1)
    trainer.board.display = False  # Four runs; we tabulate the results
    trainer.fit(model, data)
    results[name] = (d2l.num_params(model), trainer.history['val_acc'][-1])

In [ ]:
for name, (params, accuracy) in results.items():
    print(f'{name:34s} {params:>9,} parameters, '
          f'validation accuracy {accuracy:.3f}')

<!-- tbl-caption:Our roadmap on Fashion-MNIST beside the paper's. The last column is the validation accuracy after 20 epochs of the modern recipe at $64 \times 64$, rounded to the stability of repeated runs; the paper's column adds up the ImageNet top-1 gains of the rows of the table that each step covers. -->

| Step | Rows of the paper | Gain on ImageNet | Parameters | Validation accuracy |
|:--|:--|--:|--:|--:|
| Compact ResNet-18 | (ResNet-50 with the modern recipe) | | 3.35M | ≈ 94% |
| ConvNeXt blocks and stages, BN | stage ratio to fewer normalizations, separate downsampling | +3.0 | 3.38M | ≈ 94% |
| BN $\to$ LN | BN $\to$ LN | +0.1 | 3.38M | ≈ 94% |
| Patchify stem (our ConvNeXt) | patchify stem | +0.1 | 3.38M | ≈ 91.5% |

The results in the table pinpoint what transfers to $64 \times 64$ Fashion-MNIST and what does not. Keeping ResNet's overlapping stem, ConvNeXt's modernized blocks, stage ratio, and layer normalization tie with our parameter-matched `CompactResNet18` (~94%, within ~43--46M multiply-adds per image). The entire ~2.5-point drop to ~91.5% comes from the single step that was neutral on $224 \times 224$ ImageNet: the non-overlapping $4 \times 4$ stride-4 patchify stem, which jumps straight to a coarse $16 \times 16$ grid in one linear projection. Architectural choices tuned at ImageNet resolution do not transfer blindly to low-resolution domains, and a cumulative ablation immediately identifies which component to adapt.

## Limits and Open Problems

While ConvNeXt showed that pure convnets match hierarchical Vision Transformers in supervised classification under matched recipes, two active frontiers remain:

1. **Masked self-supervised pretraining:** Dropping random image patches as in masked autoencoders [@he2022masked] is natural for a sequence of patch tokens, whereas dense sliding convolutions bleed across masked holes. *ConvNeXt V2* [@woo2023convnextv2] bridges this gap with sparse convolutions and Global Response Normalization (Exercise 4).
2. **Global context:** Even a $7 \times 7$ depthwise filter is local, leaving open how ultra-large depthwise kernels [@ding2022replknet], deformable convolutions, and hybrid conv--attention stacks compare at frontier scale.

## Summary

Starting from a standard ResNet-50, ConvNeXt applied Swin-inspired upgrades one at a time—a 300-epoch modern recipe (+2.7 points), a $1{:}1{:}3{:}1$ stage ratio, a $4 \times 4$ stride-4 patchify stem, a $7 \times 7$ depthwise convolution leading a $4\times$ inverted bottleneck, a single GELU and LayerNorm per block, layer scale, and separate $2 \times 2$ stride-2 downsampling layers—reaching 82.0% on ImageNet without attention. On $64 \times 64$ Fashion-MNIST, ConvNeXt's body and LayerNorm match a parameter-matched ResNet-18 (~94%), while the coarse $4 \times 4$ patchify stem costs ~2.5 points.

## Exercises

1. **Arithmetic of the depthwise move and kernel enlargement.** In the table, just before the "Large kernels" row, each block expands $c \to 4c$ with a $1 \times 1$ convolution, filters the $4c$ channels with a $3 \times 3$ depthwise convolution, and projects $4c \to c$ with a second $1 \times 1$ convolution. The "Large kernels" step first moves the depthwise convolution to the *front* of the block (where it acts on $c$ channels instead of $4c$) and then enlarges its kernel to $7 \times 7$.
    1. Count the multiply-adds per spatial pixel of the depthwise layer before the move ($3 \times 3$ on $4c$), after the move ($3 \times 3$ on $c$), and after enlarging to $7 \times 7$ on $c$. Compare the net change against what enlarging to $7 \times 7$ *in place* on $4c$ channels would have cost.
    1. What fraction of the full `ConvNeXtBlock`'s multiply-adds per pixel ($49c + 8c^2$) does the $7 \times 7$ depthwise convolution account for at $c = 96$ and at $c = 384$?

1. **Where the parameters live in ConvNeXt.**
    1. For the `ConvNeXt` model of this section ($3{,}376{,}450$ parameters), compute the exact fraction of parameters in (i) the $7 \times 7$ depthwise convolutions, (ii) the $1 \times 1$ expansion and projection layers, (iii) the normalization and layer-scale vectors inside the blocks, (iv) the downsampling layers, and (v) the stem and head.
    1. Express the parameter count of a `ConvNeXtBlock` and a two-layer `Residual` block (that section, without a $1 \times 1$ shortcut) as functions of width $c$. By what asymptotic factor is `ConvNeXtBlock` more parameter-efficient for large $c$, and which design choice accounts for that factor?

1. [code] **Layer scale versus stochastic depth.** Under the roadmap recipe of the table, train `ConvNeXt` with layer-scale initialization $\gamma_0 = 10^{-6}$ (default), with $\gamma_0 = 1$, and with layer scale removed altogether, and compare their training curves. Both layer scale at $10^{-6}$ and stochastic depth (that section) make a deep network effectively shallower at the start of training: contrast how each mechanism works, when during training it is active, and whether it alters the evaluation-time network.

1. [code] **Global Response Normalization (ConvNeXt V2).** ConvNeXt V2 [@woo2023convnextv2] removes layer scale and inserts Global Response Normalization (GRN) right after the GELU inside each block. For a feature map $\mathbf{X}$ with $C$ channels, let $g_c = \|\mathbf{X}_c\|_2$ be the spatial $\ell_2$ norm of channel $c$ and $\bar{g} = \frac{1}{C}\sum_{c=1}^C g_c$. GRN rescales each channel by its relative response with a learnable affine transform and residual:
   $$\mathrm{GRN}(\mathbf{X})_c = \gamma_c \frac{g_c}{\bar{g} + \epsilon} \mathbf{X}_c + \beta_c + \mathbf{X}_c,$$
   where $\epsilon = 10^{-6}$ and $\boldsymbol{\gamma}, \boldsymbol{\beta}$ are initialized to zero so GRN begins as the identity. Implement GRN in a few lines of code, insert it into `ConvNeXtBlock` in place of layer scale, and compare validation accuracy under the roadmap recipe of the table.

1. [code] **Depthwise kernel size sweep.** ● Under the recipe of the table, train `ConvNeXt` with $3 \times 3$, $7 \times 7$, and $11 \times 11$ depthwise kernels (padding 1, 3, and 5) and compare parameter counts, multiply-adds, wall-clock time per epoch, and validation accuracy. Why does the wall-clock time per epoch grow much faster from $7 \times 7$ to $11 \times 11$ than the total multiply-add count would suggest?

1. [code] **2×2 architecture-by-recipe ablation.** ● The first and last rows of the table compare `CompactResNet18` and `ModernConvNeXt` under the modern AdamW/Mixup/smoothing recipe. Complete the $2 \times 2$ ablation square by training both architectures under Recipe A of that section (momentum SGD, step decay, no Mixup or label smoothing, retuning the peak learning rate). Does the modern training recipe boost both architectures equally?

[Discussions](https://d2l.discourse.group/t/36762)